# CICIDS2017 Binary Intrusion Detection Experiment

This notebook compares Logistic Regression and Random Forest for BENIGN vs MALICIOUS classification using CICIDS2017 CSV data.

## 1. Load Data

Place one or more CSV files in `data/raw/` before running this notebook. Raw data is not tracked by Git.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Image, display
from src.data_utils import clean_cicids_dataframe, load_cicids_csvs
from src.train_models import main as run_training_main

DATA_DIR = PROJECT_ROOT / "data" / "raw"
raw_df = load_cicids_csvs(DATA_DIR)
raw_df.shape

The shape above shows the number of flow rows and original columns loaded from the CSV files.

## 2. Preprocess Data

The cleaning step strips column-name whitespace, detects the label column, converts labels to binary values, keeps numeric features, and imputes missing numeric values.

In [ ]:
X, y, dataset_summary = clean_cicids_dataframe(raw_df)
display(dataset_summary)
display(y.map({0: "BENIGN", 1: "MALICIOUS"}).value_counts().rename("rows"))

The class distribution helps identify imbalance. Strong imbalance can make accuracy misleading, so precision, recall, F1-score, and false positive rate are also reported.

## 3. Train Models and Save Outputs

This calls the same CLI workflow used from the terminal. Adjust `--sample` to control runtime.

In [ ]:
import sys

old_argv = sys.argv
sys.argv = ["train_models", "--data", str(DATA_DIR), "--sample", "200000"]
try:
    run_training_main()
finally:
    sys.argv = old_argv

## 4. Metrics Table

Use F1-score and false positive rate to compare detection quality. False positives are especially important for IDS alert fatigue.

In [ ]:
import pandas as pd

metrics_path = PROJECT_ROOT / "results" / "tables" / "metrics_summary.csv"
metrics_df = pd.read_csv(metrics_path)
display(metrics_df)

## 5. Visual Evidence

The class distribution shows the dataset balance. Confusion matrices show false positives and false negatives for each model. The comparison chart summarizes model-level metrics.

In [ ]:
figure_paths = [
    PROJECT_ROOT / "results" / "figures" / "class_distribution.png",
    PROJECT_ROOT / "results" / "figures" / "confusion_matrix_logistic_regression.png",
    PROJECT_ROOT / "results" / "figures" / "confusion_matrix_random_forest.png",
    PROJECT_ROOT / "results" / "figures" / "model_metrics_comparison.png",
]

for figure_path in figure_paths:
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

## 6. Report Notes

If Random Forest has higher F1-score and lower false positive rate than Logistic Regression, the results support the hypothesis that the nonlinear model is stronger for this intrusion detection task. If the results are close, discuss possible feature separability, dataset imbalance, or leakage in the benchmark features.